# Historical RAG System


*   Sama Mohamed -- 202201867
*   Dana Amr -- 202201323

*   Zeyad sherif -- 202201220




## Dependancies

In [1]:
!pip install -q langchain_community langchain_huggingface transformers sentence_transformers pypdf chromadb bitsandbytes accelerate gradio langchain_text_splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.5 MB/s eta 0:0

In [2]:
import os
import torch
import zipfile
import glob
import gradio as gr
import torch
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

## Data Preproccessing

In [4]:
ZIP_PATH = "/content/data2.zip"
EXTRACT_PATH = "/content/extracted_data"


if not os.path.exists(EXTRACT_PATH):
    print(f"Unzipping {ZIP_PATH}...")
    try:
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_PATH)
    except FileNotFoundError:
        print("ERROR: data.zip not found! Please upload it to the Colab 'Files' tab.")
        raise

pdf_files = glob.glob(f"{EXTRACT_PATH}/**/data2/*.pdf", recursive=True)
if not pdf_files:
    pdf_files = glob.glob(f"{EXTRACT_PATH}/**/*.pdf", recursive=True)

print(f"Found {len(pdf_files)} PDF files.")

all_docs = []
for file_path in pdf_files:
    print(f"Loading: {os.path.basename(file_path)}")
    loader = PyPDFLoader(file_path)
    all_docs.extend(loader.load())

print(f"Total pages loaded: {len(all_docs)}")

Unzipping /content/data2.zip...
Found 6 PDF files.
Loading: Documentation_of_the_Sphinx.pdf
Loading: hassan_sphinx.pdf
Loading: Sphinxtemple.pdf
Loading: STONYCOLOSSUS.pdf
Loading: Histoire-photographique-du-Sphinx-EN.pdf
Loading: 1992ReconstructingtheSphinx.pdf
Total pages loaded: 506


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n", "."],
    chunk_size=500,
    chunk_overlap=50,
)
chunks = text_splitter.split_documents(all_docs)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 1658


## Embedding and storage

In [6]:
embedding_function = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True})

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_function,)

retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 3})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## RAG Pipeline

In [7]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=250,)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cuda:0


In [21]:
template = """<|system|>
You are a helpful AI assistant for a university course. Answer the user question based ONLY on the context provided below.
If the answer is not in the context, say "I don't know".

Context:
{context}</s>
<|user|>
{question}</s>
<|assistant|>
"""

prompt = ChatPromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [18]:
def ask_rag(question):
    if not question.strip():
        return "Please ask a question.", ""

    retrieved_docs = retriever.invoke(question)

    formatted_context = ""
    for i, doc in enumerate(retrieved_docs):
        source = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', '?')
        formatted_context += f"[Doc {i+1} | Source: {os.path.basename(source)} | Page {page}]\n{doc.page_content}\n\n"

    answer = rag_chain.invoke(question)

    return answer, formatted_context

## Self learning

In [19]:
def teach_model(user_feedback, question):

    if not user_feedback or len(user_feedback) < 5:
        return "Feedback too short to learn. Please provide a full sentence."

    if not question:
        return "I need to know which question this feedback applies to."

    new_doc = Document(
        page_content=f"Question: {question}\nAnswer: {user_feedback}",
        metadata={"source": "User_Correction", "type": "learned_knowledge"})

    vector_store.add_documents([new_doc])

    return "Success! I have learned this new information. Try asking the question again."

## UI

In [25]:
with gr.Blocks(title="Historical RAG System", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Historical RAG System with Self-Learning")

    with gr.Row():
        with gr.Column(scale=1):
            question_input = gr.Textbox(label="Your Question")
            submit_btn = gr.Button("Get Answer", variant="primary")
    with gr.Row():
        with gr.Column(scale=1):
            output_answer = gr.Textbox(label="AI Answer", lines=4)
            output_docs = gr.Textbox(label="Retrieved Evidence (Context)", lines=4)

    gr.Markdown("### Self-Learning Module")
    with gr.Row():
      with gr.Column(scale=1):
        feedback_input = gr.Textbox(label="Correction (If AI answer was wrong)", placeholder="Type the correct answer here")
        learn_btn = gr.Button("Teach System", variant="secondary")

    learn_status = gr.Label(label="System Status")

    submit_btn.click(
        fn=ask_rag,
        inputs=[question_input],
        outputs=[output_answer, output_docs]
    )

    learn_btn.click(
        fn=teach_model,
        inputs=[feedback_input, question_input],
        outputs=[learn_status]
    )


print("Launching Gradio UI...")
demo.launch(share=True, debug=True)

/tmp/ipython-input-260253342.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Historical RAG System", theme=gr.themes.Soft()) as demo:


Launching Gradio UI...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a006944e9231557476.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a006944e9231557476.gradio.live
